<a href="https://colab.research.google.com/github/sunsumyu/REN/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers peft trl datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00


In [3]:
import os
import torch
from google.colab import userdata
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

# 自动读取我们在 Secrets 中配置的 Hugging Face Token
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# 使用轻量高效的 Gemma 4 E4B 模型（适合 Colab T4 显存）
model_id = "google/gemma-4-e4b"

# 加载 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # 训练时建议靠右对齐

config.json:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [4]:
# 配置 4-bit 量化，大幅度降低显存占用
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # 如果硬件支持 bf16 会加速，T4 上训练器会自动处理兼容
    bnb_4bit_use_double_quant=True
)

# 加载 Gemma 4 模型本体
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # 自动将模型分配到 GPU 显存
)

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [24]:
# ==========================================
# 3. 准备并显式格式化数据集（最稳健的做法）
# ==========================================
print("⏳ 正在加载并显式格式化数据集...")
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train[:1000]")

# 直接写一个普通的清洗函数
def create_gemma_text(example):
    # 原本的数据是 "### Human: xxx ### Assistant: yyy"
    # 我们把它替换成 Gemma 4 认识的模板
    text = example["text"]
    text = text.replace("### Human:", "<bos><start_of_turn>user\n")
    text = text.replace("### Assistant:", "<end_of_turn>\n<start_of_turn>model\n")

    # 存入一个叫 "gemma_formatted_text" 的新列中
    return {"gemma_formatted_text": text}

# 显式处理数据集
formatted_dataset = dataset.map(create_gemma_text)

# ==========================================
# 4. LoRA 插件与训练器配置
# ==========================================
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",    # 我们刚刚修复过的万能钥匙
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

sft_config = SFTConfig(
    output_dir="./gemma4-finetuned",
    max_steps=60,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    report_to="none",
    max_length=512,
    dataset_text_field="gemma_formatted_text"  # ✨ 核心改动：直接告诉模型去读我们刚才处理好的列
)

# 组装 SFTTrainer（干掉 formatting_func）
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset, # ✨ 传入处理过的新数据集
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer
    # ✨ 彻底删除了容易引发 Bug 的 formatting_func
)

# ==========================================
# 5. 开始训练与保存测试
# ==========================================
print("🚀 开始微调训练！")
trainer.train()

⏳ 正在加载并显式格式化数据集...


Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_1426/3914975094.py:33: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead 

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


🚀 开始微调训练！


Step,Training Loss
10,1.860733
20,1.376843


Step,Training Loss
10,1.860733
20,1.376843
30,1.311828
40,1.252293
50,1.351476
60,1.322685


TrainOutput(global_step=60, training_loss=1.4126428445180257, metrics={'train_runtime': 2735.9976, 'train_samples_per_second': 0.175, 'train_steps_per_second': 0.022, 'total_flos': 4996146861565056.0, 'train_loss': 1.4126428445180257, 'epoch': 0.48})